# Notebook 3: Adversarial Attack Evaluation

Evaluate the effectiveness of adversarial attacks against the trained detector.

## Key Finding: Feature attacks achieve 93%+ ASR, graph attacks are ineffective.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import matplotlib.pyplot as plt

from src.dataset.loader import get_dataset
from src.dataset.preprocess import normalize_features
from src.models.detector import GCNDetector
from src.models.attack import GraphAttack, FeatureAttack
from src.utils.metrics import compute_attack_success_rate
from src.utils.visualization import plot_attack_results

In [ ]:
# Load model
checkpoint = torch.load("baseline_model.pth", map_location="cpu")
model = GCNDetector(in_channels=16, hidden_channels=64, out_channels=3)
model.load_state_dict(checkpoint["model_state"])

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# Load data
data, labels, desc = get_dataset(seed=42)
data = normalize_features(data).to(device)

target_mask = labels > 0
print(f"Targets: {target_mask.sum().item()} anomalous nodes")
print(desc)

In [ ]:
# Feature attack (primary threat)
print("=== Feature Adversarial Attack ===")
feature_attack = FeatureAttack(model, perturbation_bound=2.0, n_steps=20)
perturbed_feat, feat_info = feature_attack.attack(data, target_mask, device)
asr_feat, _ = compute_attack_success_rate(model, data, target_mask, perturbed_feat, device)
print(f"ASR: {asr_feat:.4f}")
print(f"Perturbation norm: {feat_info['perturbation_norm']:.4f}")

In [ ]:
# Graph attack across budgets
print("\n=== Graph Adversarial Attack ===")
budgets = [0.01, 0.02, 0.03, 0.05, 0.08, 0.10]
asr_graph = []

graph_attack = GraphAttack(model, budget_ratio=0.05)
for budget in budgets:
    graph_attack.budget_ratio = budget
    perturbed_data, attack_info = graph_attack.attack(data, target_mask, device)
    asr, _ = compute_attack_success_rate(model, data, target_mask, perturbed_data, device)
    asr_graph.append(asr)
    print(f"Budget={budget:.2f}: ASR={asr:.4f}, edges_added={attack_info.get('edges_added', 0)}")

In [ ]:
# Plot results
plot_attack_results(budgets, asr_graph, save_path="attack_curve.png")